# FARSHORE model server on Google Colab

Run FARSHORE's AI rival and its voice on a free Colab GPU when your laptop is too slow.

This notebook starts:

1. **Ollama** with `qwen3:4b-instruct` (Qwen3-4B-Instruct-2507). It plans every rival before a round and writes the live commentary. It is small, fast and has no slow "thinking" phase.
2. **Kokoro-82M**, a small open neural text-to-speech model, for the bard's and the rivals' voices.
3. A small **gateway** that requires an API key, so only your game can use the server.
4. A **Cloudflare quick tunnel**, which gives the gateway a public `https://….trycloudflare.com` address.

At the end it prints the lines to paste into FARSHORE's `.env`.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*. Then run every cell from top to bottom. Keep this tab open while you play: Colab stops the server when the session ends, and the tunnel address changes every time you run it again.

In [ ]:
#@title 1. Settings { display-mode: "form" }
MODEL = "qwen3:4b-instruct"  #@param ["qwen3:4b-instruct", "qwen3:1.7b", "llama3.2:3b"]
#@markdown `qwen3:4b-instruct` is the recommended model for both strategy and commentary.
ENABLE_VOICE = True  #@param {type:"boolean"}
#@markdown Leave the voice on for the bard and the rivals to speak with Kokoro. Without it the game uses the browser's built-in voices.

In [ ]:
#@title 2. Install Ollama, Kokoro and the gateway (2 to 4 minutes)
import subprocess, sys
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw espeak-ng > /dev/null", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
packages = ["fastapi", "uvicorn", "httpx", "soundfile", "numpy"] + (["kokoro>=0.9.4"] if ENABLE_VOICE else [])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True, check=True)
print("Installed.")

In [ ]:
#@title 3. Start Ollama and download the model
import os, time, urllib.request
ollama_log = open("/content/ollama.log", "w")
ollama = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "24h", "OLLAMA_NUM_PARALLEL": "2"})
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. See /content/ollama.log")
subprocess.run(["ollama", "pull", MODEL], check=True)
# Load the model onto the GPU now, so the first rival is not slow.
subprocess.run(["ollama", "run", MODEL, "Reply with the word ready."], check=True)
print(f"Ollama is serving {MODEL}.")

In [ ]:
#@title 4. Start the gateway (API key, chat proxy and voice)
import secrets
GATEWAY_KEY = secrets.token_urlsafe(24)
open("/content/gateway.py", "w").write(r"""
import io, os, secrets
import httpx, numpy as np, soundfile as sf
from fastapi import FastAPI, Header, HTTPException, Request, Response
from pydantic import BaseModel

KEY = os.environ["FARSHORE_GATEWAY_KEY"]
VOICE = os.environ.get("FARSHORE_VOICE") == "1"
app = FastAPI()
ollama = httpx.AsyncClient(base_url="http://127.0.0.1:11434", timeout=180)
pipelines = {}

def check(authorization):
    if not authorization or not secrets.compare_digest(authorization, "Bearer " + KEY):
        raise HTTPException(status_code=401, detail="Missing or wrong API key.")

@app.get("/v1/models")
async def models(authorization: str = Header(None)):
    check(authorization)
    r = await ollama.get("/v1/models")
    return Response(r.content, r.status_code, media_type="application/json")

@app.post("/v1/chat/completions")
async def chat(request: Request, authorization: str = Header(None)):
    check(authorization)
    body = await request.body()
    if len(body) > 64000:
        raise HTTPException(status_code=413, detail="Request too large.")
    r = await ollama.post("/v1/chat/completions", content=body, headers={"Content-Type": "application/json"})
    return Response(r.content, r.status_code, media_type="application/json")

class Speech(BaseModel):
    input: str
    voice: str = "bm_george"
    speed: float = 1.0
    model: str | None = None
    response_format: str | None = None

@app.post("/v1/audio/speech")
def speech(body: Speech, authorization: str = Header(None)):
    check(authorization)
    if not VOICE:
        raise HTTPException(status_code=503, detail="Voice is turned off in this notebook.")
    text = body.input.strip()[:400]
    if not text:
        raise HTTPException(status_code=400, detail="Empty input.")
    from kokoro import KPipeline
    lang = "b" if body.voice.startswith("b") else "a"
    if lang not in pipelines:
        pipelines[lang] = KPipeline(lang_code=lang)
    parts = []
    for _, _, audio in pipelines[lang](text, voice=body.voice, speed=max(0.5, min(2.0, body.speed))):
        parts.append(audio.detach().cpu().numpy() if hasattr(audio, "detach") else np.asarray(audio))
    if not parts:
        raise HTTPException(status_code=500, detail="No audio was produced.")
    buffer = io.BytesIO()
    sf.write(buffer, np.concatenate(parts), 24000, format="WAV")
    return Response(buffer.getvalue(), media_type="audio/wav")
""")
gateway_log = open("/content/gateway.log", "w")
gateway = subprocess.Popen([sys.executable, "-m", "uvicorn", "gateway:app", "--host", "127.0.0.1", "--port", "8000"],
                           cwd="/content", stdout=gateway_log, stderr=subprocess.STDOUT,
                           env={**os.environ, "FARSHORE_GATEWAY_KEY": GATEWAY_KEY, "FARSHORE_VOICE": "1" if ENABLE_VOICE else "0"})
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/docs", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("The gateway did not start. See /content/gateway.log")
print("Gateway running on port 8000.")

In [ ]:
#@title 5. Open the public tunnel
import re
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("No tunnel address yet. See /content/tunnel.log and run this cell again.")
print("Public address:", PUBLIC_URL)

In [ ]:
#@title 6. Check it works, then copy these lines into FARSHORE's .env
import json
time.sleep(4)  # A new tunnel address can take a few seconds to resolve.
def call(path, payload=None):
    request = urllib.request.Request(PUBLIC_URL + path, data=None if payload is None else json.dumps(payload).encode(),
                                     headers={"Authorization": "Bearer " + GATEWAY_KEY, "Content-Type": "application/json"})
    started = time.time()
    with urllib.request.urlopen(request, timeout=120) as response:
        return response.read(), time.time() - started
models, _ = call("/v1/models")
assert MODEL in models.decode(), "The model is not listed by the server."
reply, seconds = call("/v1/chat/completions", {"model": MODEL, "max_tokens": 40, "messages": [{"role": "user", "content": "As Polyphemus, taunt Odysseus in one short sentence."}]})
print(f"Model reply in {seconds:.1f}s:", json.loads(reply)["choices"][0]["message"]["content"])
if ENABLE_VOICE:
    audio, seconds = call("/v1/audio/speech", {"input": "Sing to me, Muse, of the man of many ways.", "voice": "bm_george"})
    print(f"Voice line in {seconds:.1f}s ({len(audio) // 1024} KB).")
print()
print("# ---- paste into FARSHORE/.env, then restart the game server ----")
print(f"FARSHORE_MODEL_URL={PUBLIC_URL}/v1/chat/completions")
print(f"FARSHORE_MODEL_NAME={MODEL}")
print(f"FARSHORE_API_KEY={GATEWAY_KEY}")
if ENABLE_VOICE:
    print(f"FARSHORE_TTS_URL={PUBLIC_URL}/v1/audio/speech")

### Notes

- **Keep this tab open.** The server stops when the Colab session ends or disconnects. Run all cells again for a new session; the address and key change each time, so update `.env` again.
- **The key matters.** The tunnel address is public. The gateway rejects any request without your key, and the key is only shown in this notebook. Don't share the printed lines.
- **Speed.** On a T4 GPU a commentary line takes well under a second and a rival strategy one to two seconds. On a CPU-only laptop the same model takes several seconds per line, which is why this notebook exists.
- **Online accounts.** For the hosted FARSHORE site, players add their own connection under Account. This notebook is for running the game yourself.